In [3]:
import pandas as pd
import glob
import re
import unicodedata
from difflib import SequenceMatcher

#=============================
# CONFIGURACIÓN
#=============================

RUTA = "Data/*.csv"
UMBRAL = 0.75

UNIT_RE = re.compile(
    r"\b(mmhg|mgdl|mg_dl|mg|dl|cm|m|lpm|hrs|hora|horas|min|seg|km|kg|porcentaje|pct)\b"
)
STOPWORDS = {
    "de", "la", "del", "y", "para", "con", "en", "por", "el", "los", "las", "un", "una", "al", "a"
}

#=============================
# FUNCIONES
#=============================

def limpiar(nombre):
    nombre = nombre.lower()
    nombre = "".join(
        c for c in unicodedata.normalize("NFD", nombre)
        if unicodedata.category(c) != "Mn"
    )
    nombre = nombre.replace("_", " ")
    nombre = nombre.replace("-", " ")
    nombre = re.sub(r"([a-z])([A-Z])", r"\1 \2", nombre)
    nombre = UNIT_RE.sub(" ", nombre)
    nombre = re.sub(r"[^a-z0-9 ]", " ", nombre)
    nombre = re.sub(r"\s+", " ", nombre).strip()
    palabras = [p for p in nombre.split() if p and p not in STOPWORDS]
    return sorted(set(palabras))


def similitud(col1, col2):
    p1 = limpiar(col1)
    p2 = limpiar(col2)

    if not p1 or not p2:
        return 0

    texto1 = " ".join(p1)
    texto2 = " ".join(p2)

    s1 = SequenceMatcher(None, texto1, texto2).ratio()
    inter = len(set(p1) & set(p2))
    union = len(set(p1) | set(p2))
    jaccard = inter / union if union else 0

    return max(s1, jaccard)


def agrupar_columnas(columnas, dfs):
    column_files = {}
    for idx, df in enumerate(dfs):
        for col in df.columns:
            column_files.setdefault(col, set()).add(idx)

    grupos = []
    for col in columnas:
        agregado = False
        for grupo in grupos:
            mismo_archivo = any(column_files[col] & column_files[otro] for otro in grupo)
            if mismo_archivo:
                continue
            if similitud(col, grupo[0]) >= UMBRAL:
                grupo.append(col)
                agregado = True
                break
        if not agregado:
            grupos.append([col])
    return grupos


def construir_mapa(grupos):
    renombrar = {}
    for grupo in grupos:
        nombre = min(grupo, key=len)
        for c in grupo:
            renombrar[c] = nombre
    return renombrar


#=============================
# LEER CSV
#=============================

archivos = glob.glob(RUTA)
if not archivos:
    raise FileNotFoundError(f"No se encontraron archivos CSV en {RUTA}")

print("Archivos encontrados:")
for archivo in archivos:
    print(" -", archivo)


dfs = [pd.read_csv(archivo) for archivo in archivos]

#=============================
# TODAS LAS COLUMNAS
#=============================

columnas = []
for df in dfs:
    columnas.extend(df.columns)
columnas = list(dict.fromkeys(columnas))

#=============================
# AGRUPAR COLUMNAS
#=============================

grupos = agrupar_columnas(columnas, dfs)
renombrar = construir_mapa(grupos)

print("\nCOLUMNAS AGRUPADAS")
for grupo in grupos:
    if len(grupo) > 1:
        print(grupo)

#=============================
# RENOMBRAR
#=============================

for i, df in enumerate(dfs):
    dfs[i] = df.rename(columns=renombrar)

#=============================
# UNIR DATASETS
#=============================

dataset = pd.concat(dfs, ignore_index=True, sort=False)
dataset = dataset.loc[:, ~dataset.columns.duplicated()]

#=============================
# GUARDAR RESULTADOS
#=============================

dataset.to_csv("dataset_unificado.csv", index=False)

mapeo = []
for grupo in grupos:
    nombre = min(grupo, key=len)
    for c in grupo:
        mapeo.append({"Columna Original": c, "Columna Final": nombre})

pd.DataFrame(mapeo).to_csv("mapeo_columnas.csv", index=False)

print("\n===================================")
print("Filas:", dataset.shape[0])
print("Columnas:", dataset.shape[1])
print("===================================")
print("\nArchivos generados:")
print(" - dataset_unificado.csv")
print(" - mapeo_columnas.csv")

Archivos encontrados:
 - Data\dataset_Diego.csv
 - Data\dataset_Matias.csv
 - Data\dataset_Mau.csv
 - Data\Dataset_Ppyo.csv
 - Data\dataset_Sesni.csv

COLUMNAS AGRUPADAS
['ID_Paciente', 'id_paciente']
['Edad', 'edad']
['Sexo', 'sexo']
['Estatura_cm', 'Estatura_m', 'estatura_m']
['Peso_kg', 'peso_kg']
['IMC', 'imc']
['Presion_Sistolica_mmHg', 'Presion_Sistolica', 'presion_sistolica', 'Presion_Arterial_Sistolica']
['Presion_Diastolica_mmHg', 'Presion_Diastolica', 'presion_diastolica']
['Frecuencia_Cardiaca_lpm', 'Frecuencia_Cardiaca', 'frecuencia_cardiaca']
['Genero', 'genero']
['Latitud', 'latitud']
['Longitud', 'longitud']
['Glucosa_mgdl', 'glucosa_mg_dl', 'Glucosa_mg_dl', 'glucosa']
['Colesterol_Total', 'colesterol_mg_dl', 'Colesterol_mg_dl', 'colesterol']
['Fumador', 'fumador']
['Actividad_Fisica', 'actividad_fisica_hrs', 'actividad_fisica']
['Diabetes', 'diabetes']
['Hipertension', 'hipertension']
['Antecedentes_Familiares', 'antecedente_familiar']

Filas: 25505
Columnas: 44

Archiv